# Train a pixel-art generator

The image pipeline in one notebook: **sprites → palette corpus → PixelGPT →
sample grids**, with a diffusion transformer trained on the same data for
comparison. Runs offline on CPU using procedurally generated sprites; swap in
`minimodel vision data prepare --dataset pixelgpt-24x24` for the real 20K
sprite corpus.

Why autoregressive for pixel art: the output vocabulary *is* a palette.
PixelGPT emits exact palette indices pixel by pixel, so there is no continuous
colour space to quantise back from — edges stay hard by construction.

In [ ]:
from pathlib import Path

from minimodel.vision.data import prepare_pixel_corpus, synthetic_sprites

work = Path("../runs/notebook-pixels")
work.mkdir(parents=True, exist_ok=True)

# 1. A corpus of symmetric sprites, quantized to a shared palette.
#    `method="auto"` detects that the sprites already use few colours and
#    keeps them exactly — lossless quantization.
sprites = synthetic_sprites(256, size=16, n_colors=12, seed=0)
stats = prepare_pixel_corpus(sprites, work / "corpus", size=16, palette_size=16)
stats

In [ ]:
# 2. Look at the data (matplotlib optional but nice here).
import numpy as np

from minimodel.vision.data import ImageCorpus, Palette

corpus = ImageCorpus(work / "corpus")
palette = Palette.load(work / "corpus" / "palette.json")
print(corpus, "|", palette)

try:
    import matplotlib.pyplot as plt

    figure, axes = plt.subplots(1, 6, figsize=(12, 2))
    for i, axis in enumerate(axes):
        axis.imshow(palette.dequantize(corpus.image(i)))
        axis.set_title(f"class {corpus.label(i)}", fontsize=8)
        axis.axis("off")
    plt.show()
except ImportError:
    print("pip install matplotlib to see the sprites inline")

In [ ]:
# 3. Train PixelGPT. 16x16 = 256 tokens per image; watch bits_per_pixel and
#    pixel_accuracy in the logs.
from minimodel.vision import PixelGPT, PixelGPTTrainer
from minimodel.vision.data import PixelSequenceDataset
from minimodel.vision.training import PixelGPTConfig

model = PixelGPT({
    "image_size": 16, "palette_size": stats["palette_size"],
    "dim": 128, "n_layers": 4, "n_heads": 4, "head_dim": 32,
    "n_kv_heads": 2, "ffn_hidden": 352, "num_classes": stats["n_classes"],
})
print(f"{model.num_parameters():,} parameters")

trainer = PixelGPTTrainer(
    model,
    PixelGPTConfig(run_name="pixelgpt", output_dir=str(work), max_steps=400,
                   batch_size=16, seq_len=256, lr=1e-3, log_every=100,
                   eval_every=0, save_every=0, resume=False),
    train_dataset=PixelSequenceDataset(work / "corpus", horizontal_flip=True),
)
result = trainer.fit()
print(f"final loss {result.final_loss:.3f}")

In [ ]:
# 4. Sample sprites. Temperature is the creativity dial; class labels steer.
import torch

from minimodel.vision.sampling import save_image_grid

samples = model.generate(16, temperature=0.8, top_p=0.9, seed=0,
                         labels=torch.zeros(16, dtype=torch.long))
path = save_image_grid(samples, work / "samples.png", palette=palette, scale=8)
print(path)

try:
    import matplotlib.pyplot as plt
    from PIL import Image

    plt.figure(figsize=(6, 6)); plt.imshow(Image.open(path)); plt.axis("off"); plt.show()
except ImportError:
    pass

In [ ]:
# 5. Sprite completion: fix the top half, let the model finish the bottom.
half = model.n_pixels // 2
originals = torch.stack([
    torch.from_numpy(corpus.image(i).astype("int64")).reshape(-1) for i in range(4)
])
completed = model.generate(4, prompt=originals[:, :half], temperature=0.8, seed=1)
save_image_grid(completed, work / "completed.png", palette=palette, scale=8)

In [ ]:
# 6. For contrast: a diffusion transformer on the RGB version of the same
#    sprites (rectified flow, 25 Euler steps at sampling time).
from minimodel.vision import DiT, DiffusionTrainer
from minimodel.vision.data import ImageDataset, prepare_image_corpus
from minimodel.vision.sampling import sample_and_save
from minimodel.vision.training import DiffusionConfig

prepare_image_corpus(sprites, work / "rgb", size=16)
dit = DiT({"image_size": 16, "patch_size": 2, "dim": 128, "depth": 4, "n_heads": 4})
DiffusionTrainer(
    dit,
    DiffusionConfig(run_name="dit", output_dir=str(work), max_steps=400,
                    batch_size=16, lr=3e-4, log_every=100, save_every=0, resume=False),
    dataset=ImageDataset(work / "rgb", horizontal_flip=True),
).fit()
sample_and_save(dit, work / "dit_samples.png", n_samples=16, n_steps=25, seed=0, scale=8)

Compare the two grids: at this data size the AR model's sprites have crisp
on-palette edges, while the diffusion samples are softer — exactly the
trade-off `docs/vision.md` describes.

## Next

- Real data: `minimodel vision data prepare --dataset pixelgpt-24x24 --mode
  palette --size 24 -o data/images/sprites`, then
  `configs/vision/pixelgpt_24x24.yaml` (the ~10M-parameter flagship).
- Class/text conditioning, editing, and the latent VAE: `docs/vision.md`.